# Regressão IBGE (Castanhal) — fluxo do curso + Random Forest

**Objetivo (como no notebook de regressão de referência):** carregar dados → **tratamento** (tipos numéricos, `fillna` com mediana) → definir **X** e **y** (layout `iloc` estilo *casas* / *plano saúde*) → **`train_test_split`** → **treinar** modelos → **prever** → **métricas** (MAE, MSE, RMSE, R²) → **gráficos** (dispersão + curva de previsão; real × previsto).

**Dados:** só **XLSX** no GitHub (`censo_castanhal/censo_castanhal/`, branch `main`), como `censo_castanhal_pipeline.ipynb`.

**Parte 1 — tabela de resultados:** o mesmo pipeline RF (`n_estimators=10` no bloco simples, `100` no multivariado) roda em **cada** DataFrame temático: `df_demografico`, `df_domicilios`, `df_educacao`, `df_trabalho_renda`, `df_unificado`, `df_features` (linhas com poucos dados ou &lt;3 colunas numéricas são ignoradas com aviso).

**Parte 2 — detalhe:** no `df_features` repetimos o fluxo completo com **gráficos** iguais ao material base (dispersão + grade de previsão; 1ª variável no teste; real × previsto com linha `y=x`).

**Pré-processamento (como no curso / `ML_pro_tcc`):** variáveis numéricas com `fillna(mediana)`; categorias nominais (`ivs_label`) viram **dummies** (`pd.get_dummies`); **`StandardScaler`** só no **Pipeline** com **regressão linear** (modelos sensíveis à escala). **Random Forest** usa matriz bruta — não exige escalonamento.

Ferramentas: `numpy`, `pandas`, `plotly`, `requests`, `sklearn` (sem funções `def`; laços `for` permitidos).


In [ ]:
!pip install -q openpyxl plotly requests "scikit-learn>=1.4"


In [ ]:
import io
import os
import re
import unicodedata
from urllib.parse import quote

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

GITHUB_REPO = os.environ.get("GITHUB_REPO", "LuanLindolfo/tcc")
XLSX_BRANCH = os.environ.get("XLSX_BRANCH", "main")
GITHUB_RAW = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{XLSX_BRANCH}"
XLSX_PATH = "censo_castanhal/censo_castanhal"

ARQUIVOS = {
    "piramide_etaria": "Censo 2022 - Pirâmide etária - Castanhal (PA).xlsx",
    "razao_sexo_envelh": "Censo 2022 - Razão de sexo e índice de envelhecimento - Castanhal (PA).xlsx",
    "populacao_residente": "Censo 2022 - População residente - Municípios.xlsx",
    "densidade_demografica": "Censo 2022 - Densidade demográfica - Municípios.xlsx",
    "caract_domicilios": "Censo 2022 - Características dos domicílios - Castanhal (PA).xlsx",
    "condicoes_ocupacao": "Censo 2022 - Condições de ocupação - Castanhal (PA).xlsx",
    "tipos_domicilio": "Censo 2022 - Tipos de domicílio - Castanhal (PA).xlsx",
    "material_paredes": "Censo 2022 - Tipo de material das paredes externas - Castanhal (PA).xlsx",
    "media_moradores": "Censo 2022 - Média de moradores por domicílio - Castanhal (PA).xlsx",
    "alfabetizacao": "Censo 2022 - Alfabetização - Castanhal (PA).xlsx",
    "nivel_instrucao": "Censo 2022 - Nível de instrução - Castanhal (PA).xlsx",
    "freq_escolar": "Censo 2022 - Taxa bruta de frequência escolar, por grupo de idade - Castanhal (PA).xlsx",
    "escolaridade_homens": "analise_escolaridade_homens_percetual.xlsx",
    "escolaridade_mulheres": "analise_escolaridade_mulheres_percetual.xlsx",
    "rendimento_percapita": "Censo 2022 - Rendimento domiciliar mensal per capita - Castanhal (PA).xlsx",
    "dist_renda": "distribuição de renda.xlsx",
    "profissoes": "profissões.xlsx",
    "taxa_atividade_pct": "taxa_atividade_percentual.xlsx",
}


In [ ]:
dados_raw = {}
for chave, arquivo in ARQUIVOS.items():
    url = f"{GITHUB_RAW}/{XLSX_PATH}/{quote(arquivo)}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dados_raw[chave] = pd.read_excel(io.BytesIO(r.content), engine="openpyxl")

dados = {}
for k, df in dados_raw.items():
    df = df.copy()
    novos = []
    for c in df.columns:
        s = unicodedata.normalize("NFKD", str(c)).encode("ascii", "ignore").decode()
        s = re.sub(r"[^\w\s]", "", s).strip().lower()
        s = re.sub(r"\s+", "_", s)
        novos.append(s)
    df.columns = novos
    num = df.select_dtypes(include="number").columns
    df[num] = df[num].fillna(df[num].median())
    dados[k] = df

frames_dem = [
    dados[k]
    for k in (
        "piramide_etaria",
        "razao_sexo_envelh",
        "populacao_residente",
        "densidade_demografica",
    )
    if k in dados and not dados[k].empty
]
df_demografico = pd.concat(frames_dem, axis=0, ignore_index=True) if frames_dem else pd.DataFrame()

frames_dom = [
    dados[k]
    for k in (
        "caract_domicilios",
        "condicoes_ocupacao",
        "tipos_domicilio",
        "material_paredes",
        "media_moradores",
    )
    if k in dados and not dados[k].empty
]
df_domicilios = pd.concat(frames_dom, axis=0, ignore_index=True) if frames_dom else pd.DataFrame()

frames_edu = [
    dados[k]
    for k in (
        "alfabetizacao",
        "nivel_instrucao",
        "freq_escolar",
        "escolaridade_homens",
        "escolaridade_mulheres",
    )
    if k in dados and not dados[k].empty
]
df_educacao = pd.concat(frames_edu, axis=0, ignore_index=True) if frames_edu else pd.DataFrame()

frames_tr = [
    dados[k]
    for k in ("rendimento_percapita", "dist_renda", "profissoes", "taxa_atividade_pct")
    if k in dados and not dados[k].empty
]
df_trabalho_renda = pd.concat(frames_tr, axis=0, ignore_index=True) if frames_tr else pd.DataFrame()

chave = None
for c in df_domicilios.columns:
    if str(c).lower() in ("setor_id", "cod_setor"):
        chave = c
        break
if chave is None:
    for c in df_domicilios.columns:
        if "setor" in str(c).lower():
            chave = c
            break

df_unificado = df_domicilios.copy()
if chave is not None:
    for df_extra, _tag in (
        (df_educacao, "_edu"),
        (df_trabalho_renda, "_tr"),
        (df_demografico, "_dem"),
    ):
        if df_extra.empty or chave not in df_extra.columns:
            continue
        overlap = [c for c in df_extra.columns if c in df_unificado.columns and c != chave]
        df_extra_ = df_extra.drop(columns=overlap, errors="ignore")
        df_unificado = df_unificado.merge(df_extra_, on=chave, how="outer")
else:
    blocos = []
    for df_blk, prefixo in (
        (df_domicilios, "dom_"),
        (df_educacao, "edu_"),
        (df_trabalho_renda, "tr_"),
        (df_demografico, "dem_"),
    ):
        if df_blk.empty:
            continue
        blocos.append(df_blk.select_dtypes(include="number").add_prefix(prefixo))
    if blocos:
        n = min(len(b) for b in blocos)
        partes = [b.iloc[:n].reset_index(drop=True) for b in blocos]
        df_unificado = pd.concat(partes, axis=1)
    else:
        df_unificado = df_domicilios.copy()

df_features = df_unificado.copy()
cols_iah = [
    c
    for c in df_features.columns
    if any(x in c for x in ["agua", "esgoto", "lixo", "energia", "saneamento"])
]
if cols_iah:
    df_features["iah"] = (df_features[cols_iah].mean(axis=1) / 100).clip(0, 1)
else:
    df_features["iah"] = 0.5

num_cols = df_features.select_dtypes(include="number").columns.tolist()
if len(num_cols) >= 2 and len(df_features) > 6:
    X_km = MinMaxScaler().fit_transform(df_features[num_cols].fillna(0))
    km = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = km.fit_predict(X_km)
    scores = X_km.mean(axis=1)
    means = {i: scores[labels == i].mean() for i in range(3)}
    ordem = sorted(means, key=lambda k: means[k])
    nomes = {ordem[0]: "baixa", ordem[1]: "media", ordem[2]: "alta"}
    df_features["ivs_label"] = [nomes[l] for l in labels]
    df_features["cluster_ocupacao"] = labels
else:
    df_features["ivs_label"] = "media"
    df_features["cluster_ocupacao"] = 0

print(
    "shapes:",
    df_demografico.shape,
    df_domicilios.shape,
    df_educacao.shape,
    df_trabalho_renda.shape,
    df_unificado.shape,
    df_features.shape,
)
df_unificado


## 1) Mesmo pipeline de ML em cada DataFrame (RF + métricas)

Para cada tabela: **tratamento** numérico (+ **dummies** de `ivs_label` quando existir) → montagem do layout estilo curso (`iloc` colunas 0–1 plano, 2 alvo, 3–18 atributos) → `train_test_split(0.3, random_state=0)` → `RandomForestRegressor(10)` no par simples e `(100)` no multivariado → **MAE, MSE, RMSE, R²** no teste.


In [ ]:
DATASETS = {
    "demografia": df_demografico,
    "domicilios": df_domicilios,
    "educacao": df_educacao,
    "trabalho_renda": df_trabalho_renda,
    "unificado": df_unificado,
    "features": df_features,
}

resultados_ml = []
for nome_tema, df in DATASETS.items():
    linha = {"tema": nome_tema}
    if df is None or df.empty or len(df) < 5:
        linha["status"] = "ignorado: menos de 5 linhas"
        resultados_ml.append(linha)
        continue
    num = df.select_dtypes(include=["number"]).copy()
    if "ivs_label" in df.columns:
        num = pd.concat([num, pd.get_dummies(df["ivs_label"], prefix="ivs", dtype=float)], axis=1)
    num = num.fillna(num.median(numeric_only=True))
    cols = [c for c in num.columns if not str(c).startswith("_pad_rf_")]
    if len(cols) < 3:
        linha["status"] = "ignorado: menos de 3 colunas numéricas"
        resultados_ml.append(linha)
        continue
    target = "iah" if "iah" in cols else cols[2]
    others = [c for c in cols if c != target]
    f0, f1 = others[0], others[1]
    feats = others[2:]
    pad_i = 0
    while len(feats) < 16:
        pad = f"_pad_rf_{pad_i}"
        num[pad] = 0.0
        feats.append(pad)
        pad_i += 1
    ordered = [f0, f1, target] + feats[:16]
    bc = num[ordered].copy()
    X_pl = bc.iloc[:, 0:1].values
    y_pl = bc.iloc[:, 1].values
    Xc = bc.iloc[:, 3:19].values
    yc = bc.iloc[:, 2].values
    Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.3, random_state=0)
    rf_s = RandomForestRegressor(n_estimators=10, random_state=0)
    rf_s.fit(X_pl, y_pl)
    rf_c = RandomForestRegressor(n_estimators=100, random_state=0)
    rf_c.fit(Xtr, ytr)
    prev = rf_c.predict(Xte)
    linha["status"] = "ok"
    linha["n_linhas"] = len(bc)
    linha["coluna_alvo"] = target
    linha["r2_plano_saude"] = rf_s.score(X_pl, y_pl)
    linha["r2_train_casas"] = rf_c.score(Xtr, ytr)
    linha["r2_test_casas"] = rf_c.score(Xte, yte)
    linha["mae_teste"] = mean_absolute_error(yte, prev)
    linha["mse_teste"] = mean_squared_error(yte, prev)
    linha["rmse_teste"] = float(np.sqrt(mean_squared_error(yte, prev)))
    linha["r2_sklearn_teste"] = r2_score(yte, prev)
    resultados_ml.append(linha)

pd.DataFrame(resultados_ml)


## 2) Detalhe em `df_features` — previsões e gráficos (como no arquivo base)

Abaixo: mesmas variáveis e comandos do curso (`X_plano_saude2`, `X_casas`, `X_teste_arvore`, `regressor_random_forest_*`, `grafico`, etc.).


In [ ]:
# Regressão (layout curso): numéricos + dummies de ivs_label (equivalente a OneHot em categorias)
num = df_features.select_dtypes(include=["number"]).copy()
num = num.fillna(num.median(numeric_only=True))
if "ivs_label" in df_features.columns:
    num = pd.concat([num, pd.get_dummies(df_features["ivs_label"], prefix="ivs", dtype=float)], axis=1)

cols = [c for c in num.columns if not str(c).startswith("_pad_rf_")]
if len(cols) < 3:
    raise ValueError("Menos de 3 colunas numéricas em df_features.")

target = "iah" if "iah" in cols else cols[2]
others = [c for c in cols if c != target]
f0, f1 = others[0], others[1]
feats = others[2:]
pad_i = 0
while len(feats) < 16:
    pad = f"_pad_rf_{pad_i}"
    num[pad] = 0.0
    feats.append(pad)
    pad_i += 1

ordered = [f0, f1, target] + feats[:16]
base_casas = num[ordered].copy()
base_casas


In [ ]:
# --- Dados no lugar de read_csv('house_prices.csv') / plano_saude2.csv — layout curso ---
base_plano_saude2 = base_casas.iloc[:, 0:2]

X_plano_saude2 = base_plano_saude2.iloc[:, 0:1].values
y_plano_saude2 = base_plano_saude2.iloc[:, 1].values

X_casas = base_casas.iloc[:, 3:19].values
y_casas = base_casas.iloc[:, 2].values

X_casas_treinamento, X_casas_teste, y_casas_treinamento, y_casas_teste = train_test_split(
    X_casas, y_casas, test_size=0.3, random_state=0
)

X_teste_arvore = np.arange(min(X_plano_saude2), max(X_plano_saude2), 0.1)
X_teste_arvore = X_teste_arvore.reshape(-1, 1)


In [ ]:
regressor_random_forest_saude = RandomForestRegressor(n_estimators=10, random_state=0)
regressor_random_forest_saude.fit(X_plano_saude2, y_plano_saude2)


In [ ]:
regressor_random_forest_saude.score(X_plano_saude2, y_plano_saude2)


In [ ]:
grafico = px.scatter(
    x=X_plano_saude2.ravel(),
    y=y_plano_saude2,
    title="IBGE (Castanhal) — RF: 1ª coluna vs 2ª + curva de previsão",
    labels={"x": "Variável previsora (col. 0)", "y": "Variável resposta (col. 1)"},
)
grafico.add_scatter(
    x=X_teste_arvore.ravel(),
    y=regressor_random_forest_saude.predict(X_teste_arvore),
    name="Regressão",
)
grafico.show()


In [ ]:
regressor_random_forest_saude.predict([[40]])


In [ ]:
regressor_random_forest_casas = RandomForestRegressor(n_estimators=100, random_state=0)
regressor_random_forest_casas.fit(X_casas_treinamento, y_casas_treinamento)


In [ ]:
regressor_random_forest_casas.score(X_casas_treinamento, y_casas_treinamento)


In [ ]:
regressor_random_forest_casas.score(X_casas_teste, y_casas_teste)


In [ ]:
previsoes = regressor_random_forest_casas.predict(X_casas_teste)
previsoes


In [ ]:
y_casas_teste


In [ ]:
pd.Series(
    {
        "MAE": mean_absolute_error(y_casas_teste, previsoes),
        "MSE": mean_squared_error(y_casas_teste, previsoes),
        "RMSE": float(np.sqrt(mean_squared_error(y_casas_teste, previsoes))),
        "R2_teste": r2_score(y_casas_teste, previsoes),
    }
)


### Comparativo: regressão linear + `StandardScaler` (como no curso para SVM/RNA)

O **Random Forest** não exige escala; modelos **lineares** sim. Mesmo `train_test_split` e mesmas matrizes `X_casas_*` / `y_casas_*` que o RF.

In [ ]:
pipe_linear = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("reg", LinearRegression()),
    ]
)
pipe_linear.fit(X_casas_treinamento, y_casas_treinamento)
previsoes_linear = pipe_linear.predict(X_casas_teste)
pd.Series(
    {
        "MAE": mean_absolute_error(y_casas_teste, previsoes_linear),
        "MSE": mean_squared_error(y_casas_teste, previsoes_linear),
        "RMSE": float(np.sqrt(mean_squared_error(y_casas_teste, previsoes_linear))),
        "R2_teste": r2_score(y_casas_teste, previsoes_linear),
    }
)

### Gráficos adicionais (teste — 1ª variável e real × previsto)


In [ ]:
x_plot = X_casas_teste[:, 0]
grafico_casas = px.scatter(
    x=x_plot,
    y=y_casas_teste,
    title="IBGE — RF (teste): 1ª variável vs real e vs previsto",
    labels={"x": "Primeira coluna de X_casas_teste", "y": "Alvo (real)"},
)
grafico_casas.add_scatter(x=x_plot, y=previsoes, name="Regressão")
grafico_casas.show()


In [ ]:
vmin = float(min(y_casas_teste.min(), previsoes.min()))
vmax = float(max(y_casas_teste.max(), previsoes.max()))
fig_rvp = px.scatter(
    x=y_casas_teste,
    y=previsoes,
    title="IBGE — Real vs previsto (teste)",
    labels={"x": "Real (alvo)", "y": "Previsto"},
)
fig_rvp.add_trace(
    go.Scatter(
        x=[vmin, vmax],
        y=[vmin, vmax],
        mode="lines",
        name="y = x",
        line=dict(dash="dash", color="gray"),
    )
)
fig_rvp.show()
